# Speaker- and room-aware speech enhancement — Colab quickstart

Runtime → Change runtime type → **T4 GPU**. Checkpoints and results go to Google Drive, so a disconnect loses nothing: re-run the cells and finished runs are skipped, unfinished ones resume.

Steps: 1) mount Drive and get the code, 2) download + prepare data (first time ~15 min), 3) smoke test, 4) baseline, 5) experiment grid, 6) results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d speaker-room-speech-enhancement || git clone https://github.com/raikar8/speaker-room-speech-enhancement.git
%cd /content/speaker-room-speech-enhancement
!git pull -q
!pip -q install -r requirements.txt

## Data
Downloads LibriSpeech train-clean-100 (6.3 GB), test-clean (0.35 GB) and RIRS_NOISES (1.3 GB). Archives are cached on Drive (`srse_archives`), so later sessions only re-extract.

In [ ]:
!python scripts/prepare_data.py --data_root /content/data --archive_dir /content/drive/MyDrive/srse_archives

## Smoke test (~5 min)
Checks causality, then trains B0 for 1 short epoch and tests on a few items. Results go to a separate `srse_smoke` folder.

In [ ]:
!python scripts/check_causality.py --config configs/experiments/M1.yaml
!python scripts/run_grid.py --experiments B0 --seeds 0 --set paths.out_dir=/content/drive/MyDrive/srse_smoke train.epochs=1 data.utts_per_speaker=5 data.test_utts=10 data.real_rir_test_utts=10

## Baseline B0 + probes
Small setting: 251 speakers × 40 utterances (~35 h of audio), 30 epochs max with early stopping. Roughly 30–60 min per run on a T4.

In [ ]:
!python scripts/run_grid.py --experiments B0 --seeds 0 --probe

## Experiment grid
Start with one seed of the core runs, then add seeds 1 and 2. Each line can be re-run safely after a disconnect.

| ID | λ_speaker | λ_room | purpose |
|---|---|---|---|
| B0 | 0 | 0 | baseline |
| S1 / S2 | 0.1 / 1 | 0 | speaker only |
| R1 / R2 | 0 | 0.1 / 1 | room only |
| M1 / M2 / M3 | 1 / 1 / 0.1 | 1 / 0.1 / 1 | both |
| C_shuf | 1 | 1 | random labels (control) |
| C_adv_spk / C_adv_room | −0.1 / 0 | 0 / −0.1 | remove the information (control) |
| O_room | – | – | true room label given (upper bound) |

In [ ]:
!python scripts/run_grid.py --experiments S2 R2 M1 --seeds 0

In [ ]:
!python scripts/run_grid.py --experiments S1 R1 M2 M3 C_shuf C_adv_spk C_adv_room O_room --seeds 0

In [ ]:
# More seeds for significance
!python scripts/run_grid.py --experiments B0 S1 S2 R1 R2 M1 M2 M3 C_shuf C_adv_spk C_adv_room O_room --seeds 1 2

## Head position and context length
Run at the best λ from the grid. `--set` changes the tap; the experiment name keeps runs apart.

In [ ]:
!python scripts/run_grid.py --experiments M1 --seeds 0 --set experiment=M1_P1 mtl.tap=P1
!python scripts/run_grid.py --experiments M1 --seeds 0 --set experiment=M1_P3 mtl.tap=P3
# Context length: re-test one trained run with fixed pooling windows (frames; 100 = 1 s)
for w in [25, 50, 100, 200, 350]:
    !python scripts/test.py --run_dir /content/drive/MyDrive/srse_runs/M1/seed0 --window {w} --tag test_w{w}

## Results

In [ ]:
!python scripts/aggregate.py --out_dir /content/drive/MyDrive/srse_runs --baseline B0
from IPython.display import Markdown, display
display(Markdown(open('/content/drive/MyDrive/srse_runs/results_test.md').read()))

In [ ]:
# Training curves of one run
import pandas as pd, matplotlib.pyplot as plt
run = '/content/drive/MyDrive/srse_runs/M1/seed0'
m = pd.read_json(f'{run}/metrics.jsonl', lines=True)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
m.plot(x='epoch', y=['train_se', 'val_se_loss'], ax=ax[0], title='SE loss')
m.plot(x='epoch', y='val_sisdri', ax=ax[1], title='val SI-SDR improvement (dB)')
m.plot(x='epoch', y=['val_spk_acc', 'val_room_acc'], ax=ax[2], title='head accuracy')
plt.tight_layout()